<a href="https://colab.research.google.com/github/Moquiuti/NLP/blob/main/TF_IDF_com_N_Grams_para_an%C3%A1lise_de_sentimentos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import nltk

nltk.download('rslp')

[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data]   Unzipping stemmers/rslp.zip.


True

In [4]:
!pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.2 MB/s eta 0:00:00


In [6]:
import pandas as pd
import nltk
from nltk import tokenize
from unidecode import unidecode

nltk.download('stopwords')
nltk.download('rslp')

df = pd.read_csv('dataset_avaliacoes.csv')

palavras_irrelevantes = nltk.corpus.stopwords.words('portuguese')

stopwords_sem_acento = [
    unidecode(palavra) for palavra in palavras_irrelevantes
]

token_pontuacao = tokenize.WordPunctTokenizer()
stemmer = nltk.RSLPStemmer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data]   Package rslp is already up-to-date!


In [7]:
from nltk import tokenize
import pandas as pd

stemmer = nltk.RSLPStemmer()
token_pontuacao = tokenize.WordPunctTokenizer()

frase_processada = []

df = pd.read_csv('dataset_avaliacoes.csv')
df.head()
for opiniao in df["avaliacao"]:
    palavras_texto = token_pontuacao.tokenize(opiniao)

    nova_frase = [
        stemmer.stem(palavra)
        for palavra in palavras_texto
        if palavra not in stopwords_sem_acento
    ]

    frase_processada.append(' '.join(nova_frase))

df["tratamento_5"] = frase_processada

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

frases = [
    "Comprei um produto ótimo",
    "Comprei um produto péssimo"
]

tfidf = TfidfVectorizer(lowercase=False, max_features=50)

caracteristicas = tfidf.fit_transform(frases)

pd.DataFrame(
    caracteristicas.todense(),
    columns=tfidf.get_feature_names_out()
)

,Comprei,produto,péssimo,um,ótimo
0,0.448321,0.448321,0.000000,0.448321,0.630099
1,0.448321,0.448321,0.630099,0.448321,0.000000


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

tfidf = TfidfVectorizer(lowercase=False, max_features=50)

tfidf_tratados = tfidf.fit_transform(df["tratamento_5"])

X_treino, X_teste, y_treino, y_teste = train_test_split(
    tfidf_tratados,
    df["sentimento"],
    random_state=4978
)

regressao_logistica = LogisticRegression(max_iter=1000)
regressao_logistica.fit(X_treino, y_treino)

acuracia_tfidf_tratados = regressao_logistica.score(X_teste, y_teste)

print(f"Acurácia do modelo: {acuracia_tfidf_tratados * 100:.2f} %")

Acurácia do modelo: 86.33 %


In [10]:
tfidf_50 = TfidfVectorizer(
    lowercase=False,
    max_features=50,
    ngram_range=(1, 2)
)

vetor_tfidf = tfidf_50.fit_transform(df["tratamento_5"])

X_treino, X_teste, y_treino, y_teste = train_test_split(
    vetor_tfidf,
    df["sentimento"],
    random_state=4978
)

regressao_logistica = LogisticRegression(max_iter=1000)
regressao_logistica.fit(X_treino, y_treino)

acuracia_tfidf_ngrams = regressao_logistica.score(X_teste, y_teste)

print(f"Acurácia do modelo com 50 features e ngrams: {acuracia_tfidf_ngrams * 100:.2f} %")

Acurácia do modelo com 50 features e ngrams: 86.33 %


In [11]:
tfidf_1000 = TfidfVectorizer(
    lowercase=False,
    max_features=1000,
    ngram_range=(1, 2)
)

vetor_tfidf_1000 = tfidf_1000.fit_transform(df["tratamento_5"])

X_treino, X_teste, y_treino, y_teste = train_test_split(
    vetor_tfidf_1000,
    df["sentimento"],
    random_state=4978
)

regressao_logistica = LogisticRegression(max_iter=1000)
regressao_logistica.fit(X_treino, y_treino)

acuracia_tfidf_ngrams_1000 = regressao_logistica.score(X_teste, y_teste)

print(f"Acurácia do modelo com 1000 features e ngrams: {acuracia_tfidf_ngrams_1000 * 100:.2f} %")

Acurácia do modelo com 1000 features e ngrams: 93.29 %


In [12]:
pesos = pd.DataFrame(
    regressao_logistica.coef_[0].T,
    index=tfidf_1000.get_feature_names_out(),
    columns=["peso"]
)

In [13]:
pesos.nlargest(50, "peso")

,peso
ótim,7.369779
excel,7.032308
bom,5.343555
ador,4.607107
perfeit,4.546882
recom,4.402985
satisfeit,4.344612
gost,3.967776
rápid,3.666547
lind,3.471849


In [14]:
pesos.nsmallest(50, "peso")

,peso
não,-6.389915
não recom,-4.559716
ruim,-3.890948
péss,-3.723793
frac,-3.688440
receb,-3.465814
defeit,-3.179270
não gost,-2.891092
vei,-2.743940
horr,-2.682500
